# Conciliação de Benefícios
**Instituto Atlântico · Departamento Pessoal**

---

Este notebook realiza a conciliação mensal de benefícios: confronta os valores cobrados pelos fornecedores com os dados de referência do DP e gera um relatório de divergências por colaborador.

**O que você vai precisar ter em mãos:**
- A fatura do fornecedor (arquivo Excel enviado mensalmente pelo fornecedor)
- A referência interna do DP (`descontos_proventos_beneficios_MMAAAA.xlsx`)

**O que este notebook entrega:**
- Relatório de divergências por colaborador, com status de cada linha
- Totais comparativos entre fatura e referência interna
- Export opcional em Excel para registro

---

### Como usar

Este notebook é organizado em **células** — blocos que você executa um a um, de cima para baixo. Para executar uma célula, clique no botão ▶ à esquerda dela ou use `Shift + Enter`.

Siga a sequência:

| Etapa | O que fazer |
|---|---|
| **Célula 1** | Preparar o ambiente — execute uma vez ao abrir o notebook |
| **Célula 2** | Carregar o sistema — execute uma vez por sessão |
| **Célula 3** | Selecionar os arquivos e confirmar |
| **Célula 4** | Visualizar o resultado da conciliação |

In [ ]:
#@title Célula 1 · Preparação do ambiente { display-mode: "form" }
#@markdown Execute uma única vez ao abrir o notebook.

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "pandas"])

import io
import pandas as pd
import openpyxl
from IPython.display import display, HTML
from google.colab import files
import ipywidgets as widgets

# Estado global compartilhado entre células
estado = {
    "uploaded":           {},   # {nome_arquivo: bytes}
    "arquivo_fatura":     None,
    "arquivo_referencia": None,
    "fornecedor":         None,
}

print("✅ Ambiente pronto.")

In [ ]:
#@title Célula 2 · Carregamento do sistema { display-mode: "form" }
#@markdown Execute uma única vez por sessão.

TOLERANCIA = 0.05  # diferença máxima em R$ para considerar OK (arredondamento)


# ════════════════════════════════════════════════════════════════════════
# PARSER: Bradesco Dental
# ════════════════════════════════════════════════════════════════════════

def parse_fatura_bradesco_dental(conteudo):
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["bradesco"]

    header_row = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if any(str(v).strip().lower() == "matricula titular" for v in row if v):
            header_row = i
            break
    if header_row is None:
        raise ValueError("Coluna 'Matricula Titular' não encontrada na aba 'bradesco'.")

    headers = list(ws.iter_rows(values_only=True))[header_row]
    idx = {}
    for i, h in enumerate(headers):
        h_norm = str(h).strip().lower() if h else ""
        if h_norm == "matricula titular":      idx["mat"]     = i
        elif h_norm == "desconto colaborador": idx["desconto"] = i
        elif h_norm in ["x", "custo"] and "custo" not in idx:
                                               idx["custo"]   = i

    registros = []
    for row in list(ws.iter_rows(values_only=True))[header_row + 2:]:
        mat   = row[idx["mat"]]      if "mat"     in idx else None
        desc  = row[idx["desconto"]] if "desconto" in idx else None
        custo = row[idx["custo"]]    if "custo"   in idx else None
        if mat and isinstance(mat, (int, float)) and desc:
            registros.append({
                "matricula":       int(mat),
                "desconto_fatura": float(desc),
                "custo_fatura":    float(custo) if custo else 0.0,
            })

    df = pd.DataFrame(registros)
    return df.groupby("matricula").agg(
        desconto_fatura=("desconto_fatura", "sum"),
        custo_fatura=("custo_fatura", "sum"),
        qtd_vidas=("matricula", "count"),
    ).reset_index()


# ════════════════════════════════════════════════════════════════════════
# PARSER: Referência interna do DP
# Corrigido: busca Fatura/Desconto/Custo somente dentro do bloco
# do fornecedor alvo, parando na primeira ocorrência.
# ════════════════════════════════════════════════════════════════════════

def parse_referencia_interna(conteudo, fornecedor_label):
    """
    Lê o arquivo de referência interna do DP (aba 'total') e extrai
    os valores esperados para o fornecedor indicado.

    Parâmetros
    ----------
    conteudo : bytes
        Conteúdo binário do arquivo xlsx.
    fornecedor_label : str
        Label do fornecedor conforme aparece no cabeçalho da planilha
        (ex: 'bradesco dental', 'unimed'). Case-insensitive.
    """
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["total"]
    rows = list(ws.iter_rows(values_only=True))

    idx_mat = idx_nome = idx_fatura = idx_desconto = idx_custo = None
    idx_bd_start = header1_row = header2_row = None

    # Localizar linha de cabeçalho 1 (onde estão 'Mat', 'Nome' e o nome do fornecedor)
    for i, row in enumerate(rows):
        vals = [str(v).strip().lower() if v else "" for v in row]
        if "mat" in vals and "nome" in vals:
            header1_row = i
            idx_mat  = next(j for j, v in enumerate(vals) if v == "mat")
            idx_nome = next(j for j, v in enumerate(vals) if v == "nome")
            for j, v in enumerate(row):
                if v and fornecedor_label.lower() in str(v).lower():
                    idx_bd_start = j
                    header2_row  = i + 1
                    break
            break

    if header2_row is None or idx_bd_start is None:
        raise ValueError(
            f"Cabeçalho do fornecedor '{fornecedor_label}' não encontrado "
            f"na aba 'total'. Verifique se o nome bate com o cabeçalho da planilha."
        )

    # Localizar Fatura / Desconto / Custo APENAS dentro do bloco do fornecedor.
    # Varre no máximo 10 colunas a partir de idx_bd_start e para ao encontrar os três.
    subheaders = rows[header2_row]
    for j in range(idx_bd_start, min(idx_bd_start + 10, len(subheaders))):
        v = subheaders[j]
        if v is None:
            continue
        v_low = str(v).strip().lower()
        if "fatura" in v_low and idx_fatura is None:
            idx_fatura = j
        elif "desconto" in v_low and idx_fatura is not None and idx_desconto is None:
            idx_desconto = j
        elif "custo" in v_low and idx_desconto is not None and idx_custo is None:
            idx_custo = j
            break  # encontrou os três — para aqui

    if None in (idx_fatura, idx_desconto, idx_custo):
        raise ValueError(
            f"Subcolunas Fatura/Desconto/Custo não encontradas para '{fornecedor_label}'. "
            f"Índices encontrados: fatura={idx_fatura}, desconto={idx_desconto}, custo={idx_custo}."
        )

    data_start = header2_row + 1
    registros  = []
    for row in rows[data_start:]:
        mat   = row[idx_mat]      if idx_mat      is not None else None
        nome  = row[idx_nome]     if idx_nome     is not None else None
        fat   = row[idx_fatura]   if idx_fatura   is not None else None
        desc  = row[idx_desconto] if idx_desconto is not None else None
        custo = row[idx_custo]    if idx_custo    is not None else None
        if mat and isinstance(mat, (int, float)):
            registros.append({
                "matricula":         int(mat),
                "nome":              nome,
                "desconto_esperado": float(desc)  if isinstance(desc,  (int, float)) else 0.0,
                "fatura_esperada":   float(fat)   if isinstance(fat,   (int, float)) else 0.0,
                "custo_esperado":    float(custo) if isinstance(custo, (int, float)) else 0.0,
            })
    return pd.DataFrame(registros)


def conciliar(df_fatura, df_referencia):
    df = df_referencia.merge(df_fatura, on="matricula", how="outer")
    df["desconto_fatura"]   = df["desconto_fatura"].fillna(0.0)
    df["desconto_esperado"] = df["desconto_esperado"].fillna(0.0)
    df["diferenca"]         = (df["desconto_fatura"] - df["desconto_esperado"]).round(4)

    def status(row):
        if abs(row["diferenca"]) <= TOLERANCIA: return "✅ OK"
        elif row["desconto_esperado"] == 0:     return "👻 Só na fatura"
        elif row["desconto_fatura"]   == 0:     return "🔍 Só na referência"
        else:                                   return "💰 Divergência de valor"

    df["status"] = df.apply(status, axis=1)
    return df


PARSERS = {
    ("bradesco", "dental"): parse_fatura_bradesco_dental,
}

# Mapeamento: chave do parser → label do fornecedor na planilha de referência
FORNECEDOR_LABELS = {
    ("bradesco", "dental"): "bradesco dental",
}


def identificar_fornecedor(nome_arquivo):
    nome_lower = nome_arquivo.lower()
    for chave, fn in PARSERS.items():
        if all(k in nome_lower for k in chave):
            return chave, fn
    return None, None


REFERENCIA_KEYWORDS = ("descontos", "proventos")

fornecedores_disponiveis = [
    " ".join(chave).title() for chave in PARSERS.keys()
]

print("✅ Sistema carregado.")
print(f"   Fornecedores disponíveis: {', '.join(fornecedores_disponiveis)}")

In [ ]:
#@title Célula 3 · Seleção de arquivos { display-mode: "form" }
#@markdown Selecione cada arquivo no botão correspondente e clique em **Confirmar** quando os dois estiverem carregados.

from IPython.display import clear_output

# ── Widgets ───────────────────────────────────────────────────────────────────

SLOT_VAZIO   = "<span style='color:#bbb;'>— aguardando arquivo</span>"
SLOT_OK      = "<span style='color:#2e7d32; font-weight:600;'>✅ {nome}</span>"
SLOT_ERRO    = "<span style='color:#c62828;'>⚠️ {nome} — {motivo}</span>"

btn_ref = widgets.Button(
    description="📋 Referência interna (DP)",
    button_style="info",
    layout=widgets.Layout(width="260px", height="44px")
)
btn_fat = widgets.Button(
    description="🧾 Fatura do fornecedor",
    button_style="info",
    layout=widgets.Layout(width="240px", height="44px")
)
btn_confirmar = widgets.Button(
    description="✔ Confirmar",
    button_style="success",
    layout=widgets.Layout(width="140px", height="36px"),
    disabled=True
)
btn_limpar = widgets.Button(
    description="✖ Limpar",
    button_style="warning",
    layout=widgets.Layout(width="100px", height="36px")
)

out_log    = widgets.Output()
out_status = widgets.Output()


def _render_status():
    ref_nome   = estado["arquivo_referencia"]
    fat_nome   = estado["arquivo_fatura"]
    fornecedor = estado["fornecedor"]

    slot_ref = (
        SLOT_OK.format(nome=ref_nome)
        if ref_nome else SLOT_VAZIO
    )
    if fat_nome and fornecedor:
        label    = ' '.join(fornecedor).title()
        slot_fat = SLOT_OK.format(nome=f"{fat_nome} ({label})")
    elif fat_nome:
        slot_fat = SLOT_ERRO.format(nome=fat_nome, motivo="fornecedor não reconhecido")
    else:
        slot_fat = SLOT_VAZIO

    pronto = bool(ref_nome and fat_nome and fornecedor)
    btn_confirmar.disabled = not pronto

    rodape = (
        "<div style='margin-top:14px; padding:10px 14px; background:#e8f5e9; "
        "border-left:3px solid #2e7d32; border-radius:4px; font-size:13px; color:#1b5e20;'>"
        "✅ <strong>Tudo pronto.</strong> Clique em <strong>Confirmar</strong> para continuar."
        "</div>"
        if pronto else ""
    )

    html = f"""
    <div style='font-family:sans-serif; font-size:14px; padding:12px 0;'>
      <table style='border-collapse:collapse; width:100%;'>
        <tr>
          <td style='padding:6px 16px 6px 0; color:#555; white-space:nowrap; vertical-align:top;'>
            📋 Referência interna
          </td>
          <td style='padding:6px 0;'>{slot_ref}</td>
        </tr>
        <tr>
          <td style='padding:6px 16px 6px 0; color:#555; white-space:nowrap; vertical-align:top;'>
            🧾 Fatura do fornecedor
          </td>
          <td style='padding:6px 0;'>{slot_fat}</td>
        </tr>
      </table>
      {rodape}
    </div>
    """
    with out_status:
        clear_output(wait=True)
        display(HTML(html))


def _carregar_referencia(b):
    with out_log:
        novos = files.upload()
    with out_log:
        clear_output(wait=True)
    for nome, conteudo in novos.items():
        if all(k in nome.lower() for k in REFERENCIA_KEYWORDS):
            estado["uploaded"][nome]     = conteudo
            estado["arquivo_referencia"] = nome
        else:
            with out_status:
                clear_output(wait=True)
                display(HTML(
                    f"<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
                    f"background:#fff3e0; border-left:3px solid #f57c00; border-radius:4px;'>"
                    f"⚠️ O arquivo <strong>{nome}</strong> não foi reconhecido como referência interna do DP.<br>"
                    f"<span style='color:#777; font-size:13px;'>O nome do arquivo deve conter "
                    f"<em>descontos</em> e <em>proventos</em>.</span></div>"
                ))
            return
    _render_status()


def _carregar_fatura(b):
    with out_log:
        novos = files.upload()
    with out_log:
        clear_output(wait=True)
    for nome, conteudo in novos.items():
        chave, _ = identificar_fornecedor(nome)
        if chave:
            estado["uploaded"][nome] = conteudo
            estado["arquivo_fatura"] = nome
            estado["fornecedor"]     = chave
        else:
            disponiveis = ", ".join(' '.join(c).title() for c in PARSERS.keys())
            with out_status:
                clear_output(wait=True)
                display(HTML(
                    f"<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
                    f"background:#fff3e0; border-left:3px solid #f57c00; border-radius:4px;'>"
                    f"⚠️ O arquivo <strong>{nome}</strong> não foi reconhecido como fatura de um fornecedor suportado.<br>"
                    f"<span style='color:#777; font-size:13px;'>Fornecedores disponíveis: <em>{disponiveis}</em>.</span></div>"
                ))
            return
    _render_status()


def _limpar(b):
    estado["uploaded"].clear()
    estado["arquivo_fatura"] = estado["arquivo_referencia"] = estado["fornecedor"] = None
    btn_confirmar.disabled = True
    with out_status:
        clear_output(wait=True)
    _render_status()


def _confirmar(b):
    btn_confirmar.disabled = True
    with out_status:
        clear_output(wait=True)
        display(HTML(
            "<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
            "background:#e8f5e9; border-left:3px solid #2e7d32; border-radius:4px; color:#1b5e20;'>"
            "✅ <strong>Confirmado.</strong> Execute a <strong>Célula 4</strong> para ver o resultado da conciliação."
            "</div>"
        ))


btn_ref.on_click(_carregar_referencia)
btn_fat.on_click(_carregar_fatura)
btn_limpar.on_click(_limpar)
btn_confirmar.on_click(_confirmar)

display(widgets.VBox([
    widgets.HTML("<div style='font-family:sans-serif; font-size:13px; color:#555; margin-bottom:6px;'>1. Selecione os arquivos</div>"),
    widgets.HBox([btn_ref, btn_fat], layout=widgets.Layout(gap="12px")),
    widgets.HTML("<div style='font-family:sans-serif; font-size:13px; color:#555; margin:12px 0 6px 0;'>2. Confirme quando os dois estiverem carregados</div>"),
    widgets.HBox([btn_limpar, btn_confirmar], layout=widgets.Layout(gap="8px")),
    out_log,
    out_status,
]))
_render_status()

In [ ]:
#@title Célula 4 · Resultado da conciliação { display-mode: "form" }
#@markdown Esta célula processa os arquivos selecionados na **Célula 3** e exibe o relatório
#@markdown de divergências por colaborador. Execute-a após confirmar os arquivos.
#@markdown
#@markdown ---
#@markdown
#@markdown **Quer salvar o resultado em Excel?** Marque a opção abaixo *antes* de executar.
#@markdown O arquivo será baixado automaticamente ao final — útil para registro ou envio por e-mail.
exportar_excel = False #@param {type:"boolean"}

from datetime import datetime as dt
from IPython.display import clear_output

# ── Validação ─────────────────────────────────────────────────────────────────
if not estado["arquivo_fatura"] or not estado["arquivo_referencia"]:
    display(HTML("""
    <div style="font-family:sans-serif; padding:12px; background:#fff3e0;
                border-left:4px solid #f57c00; border-radius:4px;">
      ⚠️ Nenhum arquivo confirmado. Execute a <strong>Célula 3</strong> primeiro.
    </div>
    """))
else:
    # ── Processamento ─────────────────────────────────────────────────────────
    chave_fornecedor = estado["fornecedor"]
    _, fn_parser     = identificar_fornecedor(estado["arquivo_fatura"])
    fornecedor_label = FORNECEDOR_LABELS[chave_fornecedor]

    df_fatura     = fn_parser(estado["uploaded"][estado["arquivo_fatura"]])
    df_referencia = parse_referencia_interna(
        estado["uploaded"][estado["arquivo_referencia"]],
        fornecedor_label=fornecedor_label,
    )
    df_result     = conciliar(df_fatura, df_referencia)

    fornecedor_display = " ".join(chave_fornecedor).title()

    # ── Métricas ──────────────────────────────────────────────────────────────
    total_ok       = (df_result["status"] == "✅ OK").sum()
    total_div      = (df_result["status"] != "✅ OK").sum()
    total_fatura_r = df_result["desconto_fatura"].sum()
    total_ref_r    = df_result["desconto_esperado"].sum()
    diff_total     = total_fatura_r - total_ref_r
    cor_div        = "#c62828" if total_div > 0 else "#2e7d32"
    cor_diff       = "#c62828" if abs(diff_total) > TOLERANCIA else "#2e7d32"

    # ── Resumo ────────────────────────────────────────────────────────────────
    display(HTML(f"""
    <div style="font-family:sans-serif; padding:20px; background:#f8f9fa;
                border-left:4px solid #1565C0; border-radius:6px; margin-bottom:16px;">
      <h3 style="margin:0 0 14px 0; color:#1a1a2e; font-size:16px;">
        📊 Conciliação — {fornecedor_display}
        <span style="font-weight:normal; color:#555; font-size:13px; margin-left:8px;">
          {dt.today().strftime('%d/%m/%Y')}
        </span>
      </h3>
      <table style="border-collapse:collapse; width:100%; font-size:14px;">
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Colaboradores analisados</td>
          <td style="font-weight:600;">{len(df_result)}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">✅ Sem divergência</td>
          <td style="font-weight:600; color:#2e7d32;">{total_ok}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">⚠️ Com divergência</td>
          <td style="font-weight:600; color:{cor_div};">{total_div}</td>
        </tr>
        <tr><td colspan="2"><hr style="margin:10px 0; border:none; border-top:1px solid #ddd;"></td></tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Total fatura (fornecedor)</td>
          <td style="font-weight:600;">R$ {total_fatura_r:.2f}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Total esperado (referência)</td>
          <td style="font-weight:600;">R$ {total_ref_r:.2f}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Diferença total</td>
          <td style="font-weight:700; color:{cor_diff};">R$ {diff_total:+.2f}</td>
        </tr>
      </table>
    </div>
    """))

    # ── Tabela detalhada ──────────────────────────────────────────────────────
    df_exibir = df_result[[
        "matricula", "nome", "qtd_vidas",
        "desconto_esperado", "desconto_fatura", "diferenca", "status"
    ]].copy()
    df_exibir.columns = [
        "Matrícula", "Nome", "Vidas",
        "Esperado (R$)", "Fatura (R$)", "Diferença (R$)", "Status"
    ]

    def _cor_linha(row):
        base = "background-color:{c}; color:#222; font-size:13px;"
        c = "#e8f5e9" if "OK" in row["Status"] else "#fff8e1"
        return [base.format(c=c)] * len(row)

    display(
        df_exibir.style
        .apply(_cor_linha, axis=1)
        .format({"Esperado (R$)": "{:.2f}", "Fatura (R$)": "{:.2f}", "Diferença (R$)": "{:+.2f}"})
        .set_table_styles([{
            "selector": "th",
            "props": [("background-color", "#1565C0"), ("color", "white"),
                      ("font-size", "13px"), ("padding", "8px 12px"),
                      ("text-align", "left")]
        }])
        .set_properties(**{"padding": "6px 12px"})
        .hide(axis="index")
    )

    # ── Export opcional ───────────────────────────────────────────────────────
    if exportar_excel:
        nome_export = f"conciliacao_{'_'.join(estado['fornecedor'])}_{dt.today().strftime('%m%Y')}.xlsx"
        df_result.to_excel(nome_export, index=False)
        files.download(nome_export)
        display(HTML(f"<p style='font-family:sans-serif; font-size:13px; color:#1565C0;'>📥 Exportado: <strong>{nome_export}</strong></p>"))
